# Credit Scoring Analysis (Converted from R to Python)
This notebook replicates the logic from the R script to score customers based on credit performance.

In [ ]:

from pathlib import Path
import pandas as pd
import numpy as np
from datetime import datetime
from dateutil.relativedelta import relativedelta


In [ ]:

def normalize_particulars(series):
    return series.str.lower().str.replace(r"[^a-z0-9]", "", regex=True)

def assign_particular_ids(df_list):
    combined = pd.concat([df[["Particulars"]] for df in df_list], ignore_index=True)
    combined["normalized"] = normalize_particulars(combined["Particulars"])
    unique_particulars = combined.drop_duplicates("normalized").copy()
    unique_particulars["particular_id"] = range(1, len(unique_particulars) + 1)
    return unique_particulars[["Particulars", "normalized", "particular_id"]]

def merge_with_ids(df, particular_dict):
    df["normalized"] = normalize_particulars(df["Particulars"])
    df = df.merge(particular_dict, on="normalized", how="left")
    df.drop(columns=["normalized"], inplace=True)
    return df

def score_creditworthiness(df):
    df["norm_payment"] = np.minimum(df["payment_ratio"], 1)
    df["norm_delay"] = 1 - np.minimum(df["avg_delay_days"], 180) / 180
    df["norm_flag"] = df["delayed"].apply(lambda x: 0 if x == 1 else 1)
    df["credit_score_norm"] = 0.5 * df["norm_payment"] + 0.3 * df["norm_delay"] + 0.2 * df["norm_flag"]
    df["credit_score"] = (100 + df["credit_score_norm"] * 800).round().astype(int)
    return df.sort_values("credit_score", ascending=False)


In [ ]:

credit_notes_path = Path("Credit_Notes_18_24_With_Tax_Amount.csv")
receipts_path = Path("Receipts_2018_to_2024.csv")
sales_register_path = Path("sales_register.csv")

credit_notes = pd.read_csv(credit_notes_path)
receipts = pd.read_csv(receipts_path)
sales_register = pd.read_csv(sales_register_path)


In [ ]:

particular_dict = assign_particular_ids([credit_notes, receipts])
credit_notes = merge_with_ids(credit_notes, particular_dict)
receipts = merge_with_ids(receipts, particular_dict)
sales_register = merge_with_ids(sales_register, particular_dict)


In [ ]:

credit_notes["Date"] = pd.to_datetime(credit_notes["Date"])
receipts["Date"] = pd.to_datetime(receipts["Date"], format="%d-%m-%y")


In [ ]:

sales_register["sale_month"] = pd.to_datetime(sales_register["YEAR"].astype(str) + "-" + sales_register["MONTH"].astype(str) + "-01")
sales_df = sales_register.groupby(["particular_id", "sale_month"], as_index=False).agg(sales_value=("sum_value", "sum"))

credit_notes["sale_month"] = credit_notes["Date"].dt.to_period("M").dt.to_timestamp()
credit_df = credit_notes.groupby(["particular_id", "sale_month"], as_index=False).agg(credit_value=("Value", "sum"))

net_sales = sales_df.merge(credit_df, on=["particular_id", "sale_month"], how="left")
net_sales["credit_value"].fillna(0, inplace=True)
net_sales["net_sales"] = net_sales["sales_value"] - net_sales["credit_value"]

receipts["receipt_month"] = receipts["Date"].dt.to_period("M").dt.to_timestamp()
receipt_df = receipts.groupby(["particular_id", "receipt_month"], as_index=False).agg(receipt_value=("Credit", "sum"))


In [ ]:

sales_receipts = net_sales.merge(receipt_df, on="particular_id", how="inner")
sales_receipts = sales_receipts[sales_receipts["receipt_month"] >= sales_receipts["sale_month"]]

sales_receipts["delay_days"] = (sales_receipts["receipt_month"] - sales_receipts["sale_month"]).dt.days
sales_receipts["delay_months"] = sales_receipts["delay_days"] / 30
sales_receipts["delayed_flag"] = (sales_receipts["delay_days"] > 180).astype(int)


In [ ]:

payment_summary = sales_receipts.groupby("particular_id").agg(
    total_net_sales=("net_sales", "sum"),
    total_receipts=("receipt_value", "sum"),
    payment_ratio=("receipt_value", lambda x: x.sum() / sales_receipts.loc[x.index, "net_sales"].sum()),
    avg_delay_days=("delay_days", lambda x: np.average(x, weights=sales_receipts.loc[x.index, "receipt_value"])),
)

payment_summary["avg_delay_months"] = payment_summary["avg_delay_days"] / 30
payment_summary["delayed"] = (payment_summary["avg_delay_days"] > 180).astype(int)
payment_summary["Delayed_Super"] = (payment_summary["avg_delay_days"] > 250).astype(int)

payment_summary_scored = score_creditworthiness(payment_summary.reset_index())
payment_summary_scored.head()
